In [3]:
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
import numpy as np
spec_file = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23'
tiff_dir  = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff'
scan_list = (17,)     # any list/tuple of scan numbers
out_vtr   = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr'    # output file

# Build with 4-circle (ZXZ: φ(Z) → χ(X) → ω(Z))
builder = RSMBuilder(
    spec_file, tiff_dir,
    selected_scans=scan_list,
    ub_includes_2pi=True,        # set False if your UB is "no-2π"
    center_is_one_based=False,)   # True if SPEC xcenter/ycenter are 1-based )


# print("buidler made")
# Compute per-pixel Q & HKL
Q_samp, hkl, intensity = builder.compute_full()
# print("buidler made Q and hkl")
h_vals = hkl[..., 0]
k_vals = hkl[..., 1]
l_vals = hkl[..., 2]

h_min, h_max = np.nanmin(h_vals), np.nanmax(h_vals)
k_min, k_max = np.nanmin(k_vals), np.nanmax(k_vals)
l_min, l_max = np.nanmin(l_vals), np.nanmax(l_vals)
builder.crop_by_positions(y_bound=(220, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped
grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(100,100,100),
   ranges=((h_min,h_max),(k_min,k_max),(l_min,l_max)),
   fuzzy=True,
   normalize="mean",
   stream=True
)

grid.shape, len(xax), len(yax), len(zax)
print(hkl[:,257, 515, 0], hkl[:,257, 515, 1], hkl[:,257, 515, 2])

Initialized QConversion area with:
  Sample Axis: ['x+', 'y+', 'z-']
  Detector Axis: ['x+']
  Beam Direction: (0, 1, 0)
  Wavelength: 1.080943 Å
  Distance: 0.781050 m
  Pixel Width: 0.000075 m
[-5.9953704 -5.9953194 -5.9952693 -5.9952135 -5.9951615 -5.9951105
 -5.9950633 -5.995011  -5.994959  -5.994905  -5.994852  -5.9948044
 -5.994751  -5.9947014 -5.994647  -5.9945974 -5.9945474 -5.994492
 -5.994443  -5.994392  -5.9943404 -5.9942904 -5.9942374 -5.994187
 -5.9941335 -5.9940825 -5.9940295 -5.9939823 -5.9939275 -5.9938745
 -5.9938245 -5.9937716 -5.9937205 -5.993674  -5.9936175 -5.993568
 -5.9935155 -5.9934673 -5.993414  -5.9933643 -5.9933105 -5.993261
 -5.9932065 -5.9931555 -5.9931016 -5.9930553 -5.9930005 -5.9929533
 -5.9928975 -5.99285   -5.992794 ] [6.0690675 6.0690875 6.0691085 6.0691223 6.0691414 6.0691614 6.0691853
 6.0692024 6.0692215 6.0692387 6.0692573 6.0692787 6.0692964 6.0693183
 6.069335  6.069355  6.069376  6.0693917 6.069413  6.0694337 6.069452
 6.069473  6.069491  6.069

In [13]:
# frame index
frm = 1

# extract H‐values for that frame
H2 = np.abs(hkl[frm, :, :, 0])

# your target H
target = 6.0485

# 1) find all pixels within a small tolerance
tol = 0.0001
ys, xs = np.where(np.abs(H2 - target) < tol)
print("pixels within ±", tol, "of", target, ":", list(zip(ys, xs, H2[ys, xs])))

# 2) find the single closest pixel
flat_idx = np.abs(H2 - target).argmin()
y0, x0 = np.unravel_index(flat_idx, H2.shape)
print(f"closest pixel at (y,x)=({y0},{x0}) with H={H2[y0,x0]:.4f}")

pixels within ± 0.0001 of 6.0485 : [(np.int64(2), np.int64(240), np.float32(6.048561)), (np.int64(3), np.int64(241), np.float32(6.048411)), (np.int64(6), np.int64(243), np.float32(6.048561)), (np.int64(7), np.int64(244), np.float32(6.048411)), (np.int64(10), np.int64(246), np.float32(6.04856)), (np.int64(11), np.int64(247), np.float32(6.0484095)), (np.int64(14), np.int64(249), np.float32(6.048558)), (np.int64(15), np.int64(250), np.float32(6.048407)), (np.int64(18), np.int64(252), np.float32(6.0485554)), (np.int64(19), np.int64(253), np.float32(6.0484037)), (np.int64(22), np.int64(255), np.float32(6.0485516)), (np.int64(26), np.int64(258), np.float32(6.0485463)), (np.int64(30), np.int64(261), np.float32(6.04854)), (np.int64(34), np.int64(264), np.float32(6.048533)), (np.int64(38), np.int64(267), np.float32(6.048525)), (np.int64(42), np.int64(270), np.float32(6.048516)), (np.int64(46), np.int64(273), np.float32(6.048506)), (np.int64(50), np.int64(276), np.float32(6.0484943)), (np.int64(

In [7]:
print(hkl[:,127, 250, 0], hkl[:,127, 250, 1], hkl[:,127, 250, 2])
# print(intensity[:,267, 515])

[-6.0986657 -6.0986285 -6.098592  -6.0985503 -6.0985117 -6.098474
 -6.0984406 -6.098402  -6.098364  -6.0983233 -6.0982842 -6.0982504
 -6.0982103 -6.0981746 -6.098134  -6.0980983 -6.098062  -6.0980206
 -6.097985  -6.0979476 -6.09791   -6.0978737 -6.097834  -6.0977974
 -6.0977583 -6.0977206 -6.0976815 -6.0976477 -6.097607  -6.097568
 -6.097532  -6.097492  -6.0974555 -6.097422  -6.0973797 -6.097344
 -6.0973053 -6.097271  -6.0972314 -6.0971956 -6.0971556 -6.09712
 -6.0970793 -6.097042  -6.097002  -6.0969696 -6.0969286 -6.096895
 -6.0968533 -6.09682   -6.0967774] [5.846058  5.846091  5.846125  5.846152  5.846184  5.8462167 5.846254
 5.846284  5.846316  5.8463464 5.8463774 5.846412  5.8464427 5.8464775
 5.8465075 5.846541  5.846575  5.8466034 5.846638  5.8466716 5.8467026
 5.8467364 5.846768  5.8468013 5.846831  5.8468637 5.8468947 5.8469315
 5.84696   5.846991  5.847026  5.8470564 5.8470902 5.847125  5.8471537
 5.8471875 5.84722   5.8472543 5.8472857 5.8473196 5.8473506 5.847383
 5.8474135 

In [5]:
import numpy as np

# assuming hkl has shape (Nf, ny, nx, 3)
h_vals = hkl[..., 0]
k_vals = hkl[..., 1]
l_vals = hkl[..., 2]

h_min, h_max = np.nanmin(h_vals), np.nanmax(h_vals)
k_min, k_max = np.nanmin(k_vals), np.nanmax(k_vals)
l_min, l_max = np.nanmin(l_vals), np.nanmax(l_vals)

print(f"H range: {h_min:.4f} → {h_max:.4f}")
print(f"K range: {k_min:.4f} → {k_max:.4f}")
print(f"L range: {l_min:.4f} → {l_max:.4f}")

H range: -5.5856 → -1.0320
K range: 1.0683 → 5.6576
L range: 0.4831 → 2.6671


In [4]:
def energy_keV_to_lambda_A(E_keV: float) -> float:
    """λ[Å] = 12.398419843320026 / E[keV]."""
    return 12.398419843320026 / float(E_keV)
energy_keV_to_lambda_A(11.47)

1.080943316767221

In [4]:
print(xax.max(), xax.min()) 
print(yax.max(), yax.min())  
print(zax.max(), zax.min())  # h, k, l

-3.4579954147338867 -3.4947268962860107
4.07067346572876 4.01666784286499
0.1912679374217987 0.17859968543052673


In [4]:
from rsm3d.data_viz import RSMNapariViewer

viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="viridis",
    rendering="attenuated_mip",  # or "mip", "translucent"
)

# launch returns the raw napari.Viewer
viewer = viz.launch()

# now add grid overlay on the *wrapper*, not on napari.Viewer
# viz.add_grid_overlay(
#     spacing=(1.0, 1.0, 1.0),   # ΔH, ΔK, ΔL
#     thickness_vox=1,
#     opacity=0.25,
#     color="white",
# )
# after compute_full & regrid_xu:
# viewer = RSMNapariViewer(grid, (xax,yax,zax), raw_intensity=builder.intensity)
# viewer(display_3d=True, use_log=True)

In [3]:
import numpy as np
import vtk
from vtk.util.numpy_support import numpy_to_vtk

# --------------------------------------------------------------------
# Replace these four lines by loading your actual grid & axes,
# e.g. via pickle / numpy.load / importing from your notebook.
# Here we demonstrate with placeholders:
#   grid shape = (nx, ny, nz) intensities
#   xax, yax, zax 1D coordinate arrays of length nx, ny, nz
# grid = np.load("rsm_g       # shape (nx, ny, nz)
# xax  = np.load("rsm_xax.npy")         # length nx
# yax  = np.load("rsm_yax.npy")         # length ny
# zax  = np.load("rsm_zax.npy")         # length nz
# --------------------------------------------------------------------

nx, ny, nz = len(xax), len(yax), len(zax)

# Build the rectilinear grid
rgrid = vtk.vtkRectilinearGrid()
rgrid.SetDimensions(nx, ny, nz)

coords_x = numpy_to_vtk(xax, deep=True)
coords_y = numpy_to_vtk(yax, deep=True)
coords_z = numpy_to_vtk(zax, deep=True)
rgrid.SetXCoordinates(coords_x)
rgrid.SetYCoordinates(coords_y)
rgrid.SetZCoordinates(coords_z)

# Flatten grid in Fortran order (Z fastest → X slowest)
flat_vals = grid.flatten(order="F")
vtk_data  = numpy_to_vtk(flat_vals, deep=True, array_type=vtk.VTK_FLOAT)
vtk_data.SetName("intensity")
rgrid.GetPointData().SetScalars(vtk_data)

# Set up GPU volume mapper (fall back to CPU mapper if needed)
mapper = vtk.vtkOpenGLGPUVolumeRayCastMapper()
mapper.SetInputData(rgrid)

# Configure volume properties (transfer functions)
vol_prop = vtk.vtkVolumeProperty()
vol_prop.SetInterpolationTypeToLinear()
vol_prop.ShadeOff()

# color transfer: map low→black, high→white
ctf = vtk.vtkColorTransferFunction()
vmin, vmax = float(np.nanmin(grid)), float(np.nanmax(grid))
ctf.AddRGBPoint(vmin, 0.0, 0.0, 0.0)
ctf.AddRGBPoint(vmax, 1.0, 1.0, 1.0)
vol_prop.SetColor(ctf)

# opacity transfer: ramp from transparent at vmin to opaque at vmax
otf = vtk.vtkPiecewiseFunction()
otf.AddPoint(vmin, 0.0)
otf.AddPoint(vmax, 1.0)
vol_prop.SetScalarOpacity(otf)

# Assemble volume actor
volume = vtk.vtkVolume()
volume.SetMapper(mapper)
volume.SetProperty(vol_prop)

# Optional: add axes actor
axes = vtk.vtkAxesActor()
axes.SetTotalLength(zax[-1] - zax[0], yax[-1] - yax[0], xax[-1] - xax[0])
axes.AxisLabelsOn()

# Renderer / render‐window / interactor
renderer = vtk.vtkRenderer()
renderer.AddVolume(volume)
renderer.AddActor(axes)
renderer.SetBackground(0.1, 0.1, 0.1)

ren_win = vtk.vtkRenderWindow()
ren_win.AddRenderer(renderer)
ren_win.SetSize(800, 800)

iren = vtk.vtkRenderWindowInteractor()
iren.SetRenderWindow(ren_win)

# Start!
iren.Initialize()
ren_win.Render()
iren.Start()


In [1]:
import numpy as np
import napari

def hkl_points_from_stacks(hkl, I,
                           frames=None,              # None → all frames; or list/array of frame indices
                           Imin=None,                # None → auto (e.g., 99th pct); or float threshold
                           stride=(1, 1),            # (y_stride, x_stride) to decimate pixels
                           sample_max=None,          # max points to keep (random downsample); None → no cap
                           finite_only=True):
    """
    Prepare HKL point cloud from per-pixel stacks.

    Parameters
    ----------
    hkl : (Nf, ny, nx, 3) float
    I   : (Nf, ny, nx)    float
    frames : iterable[int] | slice | None
    Imin : float | None
    stride : (int, int)  decimate in image space
    sample_max : int | None
    finite_only : bool

    Returns
    -------
    pos_LKH : (N, 3) float32   # (L, K, H) for napari (Z,Y,X)
    val_I   : (N,)  float32
    """
    Nf, ny, nx, _ = hkl.shape
    if frames is None:
        frames = slice(None)
    h = hkl[frames, ::stride[0], ::stride[1], 0]
    k = hkl[frames, ::stride[0], ::stride[1], 1]
    l = hkl[frames, ::stride[0], ::stride[1], 2]
    w = I  [frames, ::stride[0], ::stride[1]]

    # Flatten
    H = h.reshape(-1); K = k.reshape(-1); L = l.reshape(-1); W = w.reshape(-1)

    # Masking
    m = np.ones_like(W, dtype=bool)
    if finite_only:
        m &= np.isfinite(H) & np.isfinite(K) & np.isfinite(L) & np.isfinite(W)
    if Imin is None:
        # Auto threshold: keep top ~1% to 5% depending on sparsity
        # Use a robust heuristic: 99th percentile, fallback to 95th if degenerate
        try:
            p = np.nanpercentile(W[np.isfinite(W)], 99.0)
            if not np.isfinite(p) or p <= 0: p = np.nanpercentile(W[np.isfinite(W)], 95.0)
        except Exception:
            p = 0.0
        Imin = float(p)
    if Imin > 0:
        m &= (W >= Imin)

    H, K, L, W = H[m], K[m], L[m], W[m]

    # Optional random cap to keep interactivity
    if sample_max is not None and H.size > sample_max:
        idx = np.random.default_rng().choice(H.size, size=sample_max, replace=False)
        H, K, L, W = H[idx], K[idx], L[idx], W[idx]

    # Napari expects positions (Z,Y,X). We want axes labeled (L,K,H),
    # so build positions as (L, K, H):
    pos_LKH = np.stack([L, K, H], axis=1).astype(np.float32, copy=False)
    val_I   = W.astype(np.float32, copy=False)
    return pos_LKH, val_I

# ---------- build points ----------
# Use your data here:
# hkl = builder.hkl        # (Nf, ny, nx, 3)
# I   = builder.intensity  # (Nf, ny, nx)

pos, inten = hkl_points_from_stacks(
    builder.hkl, builder.intensity,
    frames=None,            # or e.g., range(0, builder.intensity.shape[0], 2)
    Imin=None,              # auto 99th pct; or set a value, e.g., Imin=50
    stride=(1, 1),          # increase to (2,2) or (4,4) for speed
    sample_max=1_500_000,   # cap to ~1.5M points; adjust for your GPU/CPU
)

# ---------- visualize in napari ----------
v = napari.Viewer(ndisplay=3, title="HKL point cloud")
# Map intensity to color via features DataFrame
features = {"I": inten}
pts = v.add_points(
    pos,                               # (N,3) in (L,K,H) → napari (Z,Y,X)
    name="HKL points",
    features=features,
    face_color="I",                    # use 'I' feature for colors
    # colormap="magma",                  # or 'viridis'
    size=0.05,                         # marker diameter in HKL units; tweak to your scale
    # edge_width=0.0,
    opacity=0.9,
    blending="translucent",
)
# 3D mode + nice view
v.dims.ndisplay = 3
try:
    v.reset_view()
    v.camera.angles = (30, 30, 0)
except Exception:
    pass

# Labels & scalebar (HKL are unitless)
v.dims.axis_labels = ("L", "K", "H")
v.scale_bar.visible = True
v.scale_bar.unit = ""

# Optional: add a bounding box so you see the extents
Lmin,Lmax = pos[:,0].min(), pos[:,0].max()
Kmin,Kmax = pos[:,1].min(), pos[:,1].max()
Hmin,Hmax = pos[:,2].min(), pos[:,2].max()
corners = np.array([
    [Lmin,Kmin,Hmin],[Lmin,Kmin,Hmax],[Lmin,Kmax,Hmin],[Lmin,Kmax,Hmax],
    [Lmax,Kmin,Hmin],[Lmax,Kmin,Hmax],[Lmax,Kmax,Hmin],[Lmax,Kmax,Hmax]
], dtype=np.float32)
edges = np.array([[0,1],[0,2],[0,4],[1,3],[1,5],[2,3],[2,6],[3,7],[4,5],[4,6],[5,7],[6,7]])
v.add_points(corners, name="bounds corners", size=0.03, face_color="yellow", opacity=0.8)
v.add_shapes([corners[e] for e in edges], shape_type="line", edge_color="yellow", edge_width=0.5, name="bounds box")

# Handy keys: increase/decrease point size on the fly
@v.bind_key("+")
def _bigger(viewer):
    pts.size = float(pts.size) * 1.2

@v.bind_key("-")
def _smaller(viewer):
    pts.size = float(pts.size) / 1.2

v

NameError: name 'builder' is not defined

In [2]:
# Save regridded volume to a .vtr file
import vtk
from vtk.util.numpy_support import numpy_to_vtk

# grid, (xax, yax, zax) already from:
#    grid, (xax, yax, zax) = builder.regrid_xu(...)

# out_vtr is your target filename, e.g.
#    out_vtr = '/path/to/rsm_hkl.vtr'

# Build a RectilinearGrid
nx, ny, nz = len(xax), len(yax), len(zax)
rgrid = vtk.vtkRectilinearGrid()
rgrid.SetDimensions(nx, ny, nz)

# Attach coordinate arrays
coords_x = numpy_to_vtk(xax, deep=True)
coords_y = numpy_to_vtk(yax, deep=True)
coords_z = numpy_to_vtk(zax, deep=True)
rgrid.SetXCoordinates(coords_x)
rgrid.SetYCoordinates(coords_y)
rgrid.SetZCoordinates(coords_z)

# Flatten the scalar data and add as point data
flat_int = grid.flatten(order='F')
vtk_data = numpy_to_vtk(flat_int, deep=True, array_type=vtk.VTK_FLOAT)
vtk_data.SetName('intensity')
rgrid.GetPointData().SetScalars(vtk_data)

# Write out the file
writer = vtk.vtkXMLRectilinearGridWriter()
writer.SetFileName(out_vtr)
writer.SetInputData(rgrid)
writer.Write()

print(f"Regridded volume saved to {out_vtr}")

import vtk
from vtk.util.numpy_support import numpy_to_vtk

# grid, (xax, yax, zax) already from:
#    grid, (xax, yax, zax) = builder.regrid_xu(...)

# out_vtr is your target filename, e.g.
#    out_vtr = '/path/to/rsm_hkl.vtr'

# Build a RectilinearGrid
nx, ny, nz = len(xax), len(yax), len(zax)
rgrid = vtk.vtkRectilinearGrid()
rgrid.SetDimensions(nx, ny, nz)

# Attach coordinate arrays
coords_x = numpy_to_vtk(xax, deep=True)
coords_y = numpy_to_vtk(yax, deep=True)
coords_z = numpy_to_vtk(zax, deep=True)
rgrid.SetXCoordinates(coords_x)
rgrid.SetYCoordinates(coords_y)
rgrid.SetZCoordinates(coords_z)

# Flatten the scalar data and add as point data
flat_int = grid.flatten(order='F')
vtk_data = numpy_to_vtk(flat_int, deep=True, array_type=vtk.VTK_FLOAT)
vtk_data.SetName('intensity')
rgrid.GetPointData().SetScalars(vtk_data)

# Write out the file
writer = vtk.vtkXMLRectilinearGridWriter()
writer.SetFileName(out_vtr)
writer.SetInputData(rgrid)
writer.Write()

print(f"Regridded volume saved to {out_vtr}")

Regridded volume saved to /Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr
Regridded volume saved to /Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr


In [ ]:
import numpy as np
import napari

# ---------- helpers ----------
def volume_from_grid_axes(grid, axes):
    """
    (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume plus (scale, translate) for napari.
    Uses average spacing for the linear world transform; exact coords come from axes below.
    """
    xax, yax, zax = [np.asarray(a) for a in axes]
    nx, ny, nz = len(xax), len(yax), len(zax)
    if grid.shape != (nx, ny, nz):
        raise ValueError(f"grid shape {grid.shape} != ({nx},{ny},{nz}) from axes")

    vol = grid.transpose(2, 1, 0).copy()  # (Z,Y,X)

    def _avg_step(a): return float(np.diff(a).mean()) if len(a) > 1 else 1.0
    dx, dy, dz = _avg_step(xax), _avg_step(yax), _avg_step(zax)

    translate = (float(zax[0]), float(yax[0]), float(xax[0]))  # (Z,Y,X)
    scale     = (dz, dy, dx)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return vol, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

def index_to_axis_value(ax, idx):
    """
    Map a (possibly fractional) data index -> exact world coordinate using your axis array.
    Works for non-uniform axes.
    """
    n = len(ax)
    if n == 0:
        return np.nan
    if idx <= 0:
        return float(ax[0])
    if idx >= n - 1:
        return float(ax[-1])
    i0 = int(np.floor(idx))
    t  = float(idx - i0)
    return float((1.0 - t) * ax[i0] + t * ax[i0 + 1])

# ---------- build napari viewer ----------
# Assumes you already have: grid, (xax, yax, zax)
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

v = napari.Viewer(ndisplay=3, title="RSM viewer")

vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])  # quick sensible contrast
img_layer = v.add_image(
    vol_log,
    name="RSM (log1p)",
    colormap="viridis",
    rendering="attenuated_mip",
    blending="translucent",
    opacity=1.0,
    scale=scale,          # linear world transform (approx if non-uniform axes)
    translate=translate,
    # contrast_limits=(float(lo), float(hi)),
)

# UI niceties
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
v.scale_bar.unit = ""              # set to "Å⁻¹" if you’re viewing Q-space
v.dims.axis_labels = ("L", "K", "H")  # for HKL; use ("Qz","Qy","Qx") for Q-space

# ---------- super-thin corners & outline (world coordinates) ----------
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

# tiny corner markers: << 1 voxel (use isotropic size for napari stable)
voxel = np.array(scale, dtype=float)           # (dz, dy, dx)
corner_size = min(voxel) * 0.15                # 0.15 of smallest voxel size
corner_sizes = np.full(8, corner_size)         # (8,) isotropic marker size
v.add_points(
    corners_world,
    name="Outline corners",
    size=corner_sizes,      # isotropic (length 8)
    face_color="red",
    opacity=0.9,
    blending="additive",
)

# hairline outline
box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]
v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.05,               # sub-pixel hairline
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# slim world-axis vectors
Lx, Ly, Lz = xmax - xmin, ymax - ymin, zmax - zmin
axes_len = 0.10 * max(Lx, Ly, Lz)
origin = np.array([zmin, ymin, xmin], dtype=float)
vectors = np.stack([
    np.vstack([origin, origin + np.array([axes_len, 0, 0])]),  # +Z
    np.vstack([origin, origin + np.array([0, axes_len, 0])]),  # +Y
    np.vstack([origin, origin + np.array([0, 0, axes_len])]),  # +X
], axis=0)
# v.add_vectors(
#     vectors,
#     name="World axes",
#     edge_color=["cyan", "lime", "magenta"],
#     edge_width=0.75,
#     blending="translucent_no_depth",
# )

# ---------- live coordinate HUD (H, K, L + intensity under cursor) ----------
# Uses exact coordinate mapping from axis arrays (works for non-uniform axes).
# If napari version has viewer.text_overlay, we’ll use it; otherwise we print to console.
def on_mouse_move(viewer, event):
    pos_world = viewer.cursor.position
    if pos_world is None:
        return
    # data indices in (Z,Y,X) for this layer
    zi, yi, xi = img_layer.world_to_data(pos_world)
    # intensity sample (linear intensity from 'volume')
    I = np.nan
    zi_i, yi_i, xi_i = int(np.round(zi)), int(np.round(yi)), int(np.round(xi))
    if (0 <= zi_i < volume.shape[0]) and (0 <= yi_i < volume.shape[1]) and (0 <= xi_i < volume.shape[2]):
        I = float(volume[zi_i, yi_i, xi_i])

    # exact HKL (or Q) using axis arrays
    H = index_to_axis_value(xax, xi)
    K = index_to_axis_value(yax, yi)
    L = index_to_axis_value(zax, zi)
    text = f"H={H:.4f}   K={K:.4f}   L={L:.4f}    I={I:.3g}"

    if hasattr(viewer, "text_overlay") and viewer.text_overlay is not None:
        overlay = viewer.text_overlay
        overlay.visible = True
        overlay.position = 'top_left'
        overlay.color = 'white'
        overlay.font_size = 12
        overlay.text = text
    else:
        print(text, end="\r")

v.mouse_move_callbacks.append(on_mouse_move)

# Optional: press 'C' to toggle the HUD
@v.bind_key('C')
def _toggle_coords(viewer):
    if hasattr(viewer, "text_overlay") and viewer.text_overlay is not None:
        viewer.text_overlay.visible = not viewer.text_overlay.visible

print("Uniform voxel spacing (for the linear transform):", is_uniform)
v

Uniform voxel spacing (for the linear transform): True


Viewer(camera=Camera(center=(np.float64(0.6176714897155762), np.float64(3.2585725784301767), np.float64(2.270174503326418)), zoom=np.float64(20.246117478503688), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(0.0, 0.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=3, order=(0, 1, 2), axis_labels=('L', 'K', 'H'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(-10.377758026123047), stop=np.float64(11.6131010055542), step=np.float64(0.11050682930491078)), RangeTuple(start=np.float64(-5.43487548828125), stop=np.float64(11.952020645141602), step=np.float64(0.08737133735388368)), RangeTuple(start=np.float64(-11.736214637756348), stop=np.float64(16.27656364440918), step=np.float64(0.1407677300611333))), margin_left=(0.0

In [ ]:
import numpy as np
import napari

# ---------- helpers ----------
def volume_from_grid_axes(grid, axes):
    """
    (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume and (scale, translate) for napari.
    Uses average spacing for the linear world transform (exact coords still come from axes).
    """
    xax, yax, zax = [np.asarray(a) for a in axes]
    nx, ny, nz = len(xax), len(yax), len(zax)
    if grid.shape != (nx, ny, nz):
        raise ValueError(f"grid shape {grid.shape} != ({nx},{ny},{nz}) from axes")

    vol = grid.transpose(2, 1, 0).copy()  # (Z,Y,X)

    def _avg_step(a): return float(np.diff(a).mean()) if len(a) > 1 else 1.0
    dx, dy, dz = _avg_step(xax), _avg_step(yax), _avg_step(zax)

    translate = (float(zax[0]), float(yax[0]), float(xax[0]))  # (Z,Y,X)
    scale     = (dz, dy, dx)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return vol, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

# ---------- build napari viewer (FORCE VOLUME MODE) ----------
# Assumes you already have: grid, (xax, yax, zax)
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

# Sanity: ensure truly 3-D
assert volume.ndim == 3, f"Expected 3-D, got {volume.ndim}D"
nz, ny, nx = volume.shape
assert nz > 1 and ny > 1 and nx > 1, f"One dimension is singleton: (nz,ny,nx)={volume.shape}"

v = napari.Viewer(title="RSM viewer")
# force viewer to 3-D
v.dims.ndisplay = 3

vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])

img_layer = v.add_image(
    vol_log,
    name="RSM (log1p)",
    scale=scale,                # (Z,Y,X)
    translate=translate,        # (Z,Y,X)
    contrast_limits=(float(lo), float(hi)),
)

# Make absolutely sure it's a VOLUME, not a slicing plane
if hasattr(img_layer, "depiction"):
    img_layer.depiction = "volume"     # napari >= 0.5
# Pick a 3-D renderer
if hasattr(img_layer, "rendering"):
    img_layer.rendering = "attenuated_mip"

# Make sure the viewer is in 3-D after adding the layer
v.dims.ndisplay = 3

# Frame the whole volume and set a non-orthogonal camera so it is clearly 3-D
try:
    v.reset_view()
    v.camera.angles = (30, 30, 0)  # yaw, pitch, roll (deg) for a nice 3-D angle
    v.camera.zoom = 1.0
except Exception:
    pass

# UI niceties
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
v.scale_bar.unit = ""             # "Å⁻¹" if Q-space
v.dims.axis_labels = ("L", "K", "H")

# ---------- draw thin outline & corners (unchanged functionality, very light) ----------
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

voxel = np.array(scale, dtype=float)           # (dz, dy, dx)
corner_size = float(min(voxel) * 0.15)
v.add_points(
    corners_world,
    name="Outline corners",
    size=np.full(8, corner_size),  # isotropic markers
    face_color="red",
    # edge_width=0,
    opacity=0.9,
    blending="additive",
)

box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]
v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.1,               # pixel units (hairline)
    # edge_width_is_relative=False,
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# ---------- diagnostics: confirm world extent covers the whole volume ----------
try:
    # data extent (indices) and world extent (coords)
    print("Data extent (Z,Y,X):", img_layer.extent.data)    # ( (zmin,zmax), (ymin,ymax), (xmin,xmax) ) in index space
    print("World extent (Z,Y,X):", img_layer.extent.world)  # same, transformed by scale/translate
except Exception:
    pass

print("Volume shape (Z,Y,X):", volume.shape)
print("Uniform voxel spacing (for linear transform):", is_uniform)

v

NameError: name 'grid' is not defined